In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, DoubleType
from pyspark.ml.feature import VectorAssembler
import joblib
import numpy as np

# -------------------------
# 1️⃣ Spark session
# -------------------------
spark = SparkSession.builder \
    .appName("KafkaSparkXGBCountFraud") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# -------------------------
# 2️⃣ Schema
# -------------------------
schema = StructType(
    [StructField("Time", DoubleType(), True)] +
    [StructField(f"V{i}", DoubleType(), True) for i in range(1, 29)] +
    [StructField("Amount", DoubleType(), True)]
)

# -------------------------
# 3️⃣ Kafka stream
# -------------------------
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "topic") \
    .option("startingOffsets", "earliest") \
    .load()

# -------------------------
# 4️⃣ Parse JSON
# -------------------------
parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

# -------------------------
# 5️⃣ Feature Engineering
# -------------------------
feature_cols = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
features_df = assembler.transform(parsed_df)

# -------------------------
# 6️⃣ Load model
# -------------------------
xgb_model = joblib.load("../../models/xgboost_fraud_model2.pkl")

# -------------------------
# 7️⃣ Global counters
# -------------------------
total_seen = 0
total_predicted_fraud = 0

# -------------------------
# 8️⃣ Prediction + counting
# -------------------------
def predict_and_count(batch_df, batch_id):
    global total_seen, total_predicted_fraud

    batch_count = batch_df.count()
    if batch_count == 0:
        return

    pdf = batch_df.select("features").toPandas()
    X = np.array(pdf["features"].tolist())

    probas = xgb_model.predict_proba(X)[:, 1]

    # ⚠️ Threshold
    predictions = (probas > 0.9).astype(int)

    batch_fraud = predictions.sum()

    total_seen += batch_count
    total_predicted_fraud += batch_fraud

    pdf["fraud_proba"] = probas
    pdf["prediction"] = predictions
    pdf["fraud"] = np.where(predictions == 1, "FRAUD", "")

    # 🔴 Keep only fraud rows
    fraud_rows = pdf[pdf["prediction"] == 1]

    print("-------------------------------------------------------")
    print(f"\n========== BATCH {batch_id} ==========")

    # ✅ Show fraud rows ONLY if they exist
    if batch_fraud > 0:
        print("🚨 FRAUD DETECTED:")
        print(fraud_rows)
    else:
        print("✅ No fraud detected in this batch")

    print(f"Rows in batch: {batch_count}")
    print(f"Fraud predicted in batch: {batch_fraud}")
    print(f"TOTAL rows seen: {total_seen}")
    print(f"TOTAL fraud predicted: {total_predicted_fraud}")

    print("-------------------------------------------------------")


# -------------------------
# 9️⃣ Start stream
# -------------------------
query = features_df.writeStream \
    .foreachBatch(predict_and_count) \
    .trigger(processingTime="1 second") \
    .start()

query.awaitTermination()


26/01/06 01:47:10 WARN Utils: Your hostname, DESKTOP-DD5N9SQ resolves to a loopback address: 127.0.1.1; using 172.17.138.117 instead (on interface eth0)
26/01/06 01:47:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/souhail/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/souhail/.ivy2/cache
The jars for the packages stored in: /home/souhail/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e611b90e-918c-47f3-946b-c90a009b2d66;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 2957ms :: artifacts dl 106ms

-------------------------------------------------------

========== BATCH 0 ==========
✅ No fraud detected in this batch
Rows in batch: 368
Fraud predicted in batch: 0
TOTAL rows seen: 368
TOTAL fraud predicted: 0
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1 ==========
✅ No fraud detected in this batch
Rows in batch: 120
Fraud predicted in batch: 0
TOTAL rows seen: 488
TOTAL fraud predicted: 0
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 2 ==========
✅ No fraud detected in this batch
Rows in batch: 18
Fraud predicted in batch: 0
TOTAL rows seen: 506
TOTAL fraud predicted: 0
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 3 ==========
✅ No fraud detected in this batch
Rows in batch: 25
Fraud predicted in batch: 0
TOTAL rows seen: 531
TOT

-------------------------------------------------------

========== BATCH 6 ==========
✅ No fraud detected in this batch
Rows in batch: 35
Fraud predicted in batch: 0
TOTAL rows seen: 631
TOTAL fraud predicted: 0
-------------------------------------------------------


...

-------------------------------------------------------

========== BATCH 1761 ==========
✅ No fraud detected in this batch
Rows in batch: 11
Fraud predicted in batch: 0
TOTAL rows seen: 18931
TOTAL fraud predicted: 1
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1762 ==========
✅ No fraud detected in this batch
Rows in batch: 18
Fraud predicted in batch: 0
TOTAL rows seen: 18949
TOTAL fraud predicted: 1
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1763 ==========
✅ No fraud detected in this batch
Rows in batch: 12
Fraud predicted in batch: 0
TOTAL rows seen: 18961
TOTAL fraud predicted: 1
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1764 ==========
✅ No fraud detected in this batch
Rows in batch: 11
Fraud predicted in batch: 0
TOTAL ro

-------------------------------------------------------

========== BATCH 1769 ==========
✅ No fraud detected in this batch
Rows in batch: 9
Fraud predicted in batch: 0
TOTAL rows seen: 19022
TOTAL fraud predicted: 1
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1770 ==========
✅ No fraud detected in this batch
Rows in batch: 27
Fraud predicted in batch: 0
TOTAL rows seen: 19049
TOTAL fraud predicted: 1
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1771 ==========
✅ No fraud detected in this batch
Rows in batch: 20
Fraud predicted in batch: 0
TOTAL rows seen: 19069
TOTAL fraud predicted: 1
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1772 ==========
✅ No fraud detected in this batch
Rows in batch: 13
Fraud predicted in batch: 0
TOTAL row

-------------------------------------------------------

========== BATCH 1860 ==========
✅ No fraud detected in this batch
Rows in batch: 19
Fraud predicted in batch: 0
TOTAL rows seen: 19986
TOTAL fraud predicted: 1
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1861 ==========
✅ No fraud detected in this batch
Rows in batch: 49
Fraud predicted in batch: 0
TOTAL rows seen: 20035
TOTAL fraud predicted: 1
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1862 ==========
✅ No fraud detected in this batch
Rows in batch: 11
Fraud predicted in batch: 0
TOTAL rows seen: 20046
TOTAL fraud predicted: 1
-------------------------------------------------------
-------------------------------------------------------

========== BATCH 1863 ==========
✅ No fraud detected in this batch
Rows in batch: 11
Fraud predicted in batch: 0
TOTAL ro

-------------------------------------------------------

========== BATCH 2257 ==========
🚨 FRAUD DETECTED:
                                            features  fraud_proba  prediction  \
0  [169351.0, -0.676142670593205, 1.1263660623459...     0.999240           1   
1  [169966.0, -3.11383160738512, 0.58586417180689...     0.999959           1   
2  [169966.0, -3.11383160738512, 0.58586417180689...     0.999959           1   
3  [169966.0, -3.11383160738512, 0.58586417180689...     0.999959           1   
4  [169966.0, -3.11383160738512, 0.58586417180689...     0.999959           1   
5  [169966.0, -3.11383160738512, 0.58586417180689...     0.999959           1   
6  [169966.0, -3.11383160738512, 0.58586417180689...     0.999959           1   
7  [170348.0, 1.9919760961759, 0.158475887304227,...     0.994153           1   
8  [170348.0, 1.9919760961759, 0.158475887304227,...     0.994153           1   
9  [170348.0, 1.9919760961759, 0.158475887304227,...     0.994153           1   



ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=58>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/souhail/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline(